## High Volatility feature

In [1]:
import pandas as pd
import numpy as np

print("1. Loading V2.5 15-Minute Dataset...")

df = pd.read_csv('V2.5_15min_features.csv')

df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

print("2. Defining 'High Volatility' Threshold...")
#use rolling std to define high volatility
# noted:
df['price_roll_std_6h'] = df['price'].rolling(window=24).std()

# delete rows with NaN values in 'price_roll_std_6h' column
df = df.dropna(subset=['price_roll_std_6h']).copy()

vol_threshold = df['price_roll_std_6h'].quantile(0.85)
print(f"   -> Top 15% Volatility Threshold: {vol_threshold:.2f}")

print("3. Creating the Binary Classification Target (0 or 1)...")
df['is_high_volatility'] = (df['price_roll_std_6h'] >= vol_threshold).astype(int)
print(df['is_high_volatility'].value_counts(normalize=True) * 100)


1. Loading V2.5 15-Minute Dataset...


ValueError: Mixed timezones detected. Pass utc=True in to_datetime or tz='UTC' in DatetimeIndex to convert to a common timezone.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("4. Selecting Safe Features for the Classifier...")

# temp_lag_4 = one of the lag features, which is safe to use for classification
vol_features = ['temp', 'wind_speed', 'wind_direction_deg', 'temp_lag_4',
                'hour', 'day_of_week', 'month']

X_vol = df[vol_features]
y_vol = df['is_high_volatility']

# to make sure there are no NaN values in the features, we can drop rows with NaN values
mask = X_vol.notna().all(axis=1)
X_vol, y_vol = X_vol[mask], y_vol[mask]

X_vol_train, X_vol_test, y_vol_train, y_vol_test = train_test_split(
    X_vol, y_vol, test_size=0.2, shuffle=False
)

print("5. Initializing and Training the XGBoost Risk Classifier...")
risk_model = XGBClassifier(
    n_estimators=100,       # 100棵警戒树
    learning_rate=0.05,     
    max_depth=5,            # depth of each tree,as a hyperparameter, can be tuned
    objective='binary:logistic',  # binary classification
    random_state=42         # lock the random seed for reproducibility
)
risk_model.fit(X_vol_train, y_vol_train)

print("6. Evaluating the Risk Classifier (NEW STEP)...")
y_pred = risk_model.predict(X_vol_test)
print('Test accuracy:', round(accuracy_score(y_vol_test, y_pred), 4))
print(classification_report(y_vol_test, y_pred))

print("7. Extracting the High-Volatility Probabilities...")
# predict_proba returns [safe probability, high-volatility probability], take the second column [:, 1]
predicted_probabilities = risk_model.predict_proba(X_vol)[:, 1]
df['high_volatility_prob'] = predicted_probabilities

print("   -> New feature 'high_volatility_prob' created!")
df[['datetime', 'price', 'is_high_volatility', 'high_volatility_prob']].tail(10)


In [ ]:
print("8. Saving the new V3 Matrix (V2.5 + Risk Feature)...")
#save
save_path = 'V3.0_15min_Risk_Enhanced_Dataset.csv'
df.to_csv(save_path, index=False)
print(f"Matrix saved to: {save_path}")